# CAASPP School-Level DFS — companion notebook

Calculates **Smarter Balanced-only** Distance from Standard by elementary school, including **Mariposa**. This is not official California School Dashboard DFS (which can include alternate assessment/participation adjustments). Requires CAASPP **2024–2026 fixed-width ASCII** districtwide All Student Groups TXT or ZIP files. Older file layouts and delimited files are not yet supported.

Sources: [CAASPP file layout](https://caaspp-elpac.ets.org/caaspp/ResearchFileFormatSB?lstTestYear=2026) and [Smarter Balanced cut scores](https://technicalreports.smarterbalanced.org/scoring_specs/_book/scoringspecs.html).


In [ ]:
#@title Step 1: Upload fixed-width ASCII TXT/ZIP files
from google.colab import files
uploaded=files.upload()
if not uploaded: raise ValueError('Please upload at least one .txt or .zip file')


In [ ]:
#@title Step 2: Calculate school-level DFS
from zipfile import ZipFile
from io import BytesIO
import pandas as pd
CUTS={'01': {'03':2432,'04':2473,'05':2502}, '02': {'03':2436,'04':2485,'05':2528}}
SUBJECT={'01':'ELA','02':'Math'}
COUNTY_CODE='19' #@param {type:'string'}
DISTRICT_CODE='64683' #@param {type:'string'}
def f(s,a,b): return s[a-1:b].strip()
def lines_of_uploads(uploaded):
    for name, blob in uploaded.items():
        if name.lower().endswith('.zip'):
            with ZipFile(BytesIO(blob)) as z:
                for member in z.namelist():
                    if member.lower().endswith('.txt') and not member.startswith('__MACOSX'):
                        yield member, z.read(member).decode('utf-8-sig',errors='replace').splitlines()
        elif name.lower().endswith('.txt'):
            yield name,blob.decode('utf-8-sig',errors='replace').splitlines()
        else: raise ValueError(f'Unsupported input: {name}')
rows=[]
for source,lines in lines_of_uploads(uploaded):
    for line in lines:
        if not line.strip(): continue
        if len(line)<201 or not f(line,121,124).isdigit():
            raise ValueError(f'{source}: expected 2024+ fixed-width named ASCII layout')
        if f(line,115,116)!='07' or f(line,128,130)!='001': continue
        grade,test=f(line,131,132),f(line,126,127)
        if grade not in ('03','04','05') or test not in CUTS: continue
        if f(line,1,2)!=COUNTY_CODE.zfill(2) or f(line,3,7)!=DISTRICT_CODE.zfill(5): continue
        try:
            count=int(f(line,151,159))
            mean=float(f(line,160,165))
        except ValueError: continue
        if count<=0: continue
        rows.append({'Test Year':int(f(line,121,124)),'School Code':f(line,48,54),
                     'School':f(line,55,114),'Grade':int(grade),'Subject':SUBJECT[test],
                     'Students With Scores':count,'Mean Scale Score':mean,
                     'Level 3 Cutoff':CUTS[test][grade], 'DFS':mean-CUTS[test][grade], 'Source File':source})
detail=pd.DataFrame(rows)
if detail.empty: raise ValueError('No matching school-level grades 3–5 rows.')
key=['Test Year','School Code','Grade','Subject']
if detail.duplicated(key).any(): raise ValueError('Duplicate data; upload only one copy of each year.')
detail['weighted']=detail['DFS']*detail['Students With Scores']
school=(detail.groupby(['Test Year','School Code','School','Subject'],as_index=False)
        .agg(Students_With_Scores=('Students With Scores','sum'),
             Weighted_Total=('weighted','sum'),Grades_Included=('Grade','nunique')))
school['DFS']=(school['Weighted_Total']/school['Students_With_Scores']).round(1)
school['Complete_3_5']=school['Grades_Included'].eq(3)
school=school.drop(columns=['Weighted_Total']).sort_values(['Test Year','Subject','School'])
detail.drop(columns=['weighted']).to_csv('caaspp_dfs_grade_detail.csv',index=False)
school.to_csv('caaspp_dfs_school_summary.csv',index=False)
print('Calculated Smarter Balanced DFS (not official Dashboard DFS). Includes Mariposa.')
display(school[['Test Year','School','Subject','Students_With_Scores','Grades_Included','Complete_3_5','DFS']])


In [ ]:
#@title Step 3: Download tables
from google.colab import files
files.download('caaspp_dfs_grade_detail.csv')
files.download('caaspp_dfs_school_summary.csv')


## Interpretation

Plot DFS by Test Year for each school and subject. Check Complete_3_5 before treating a result as a comparable three-grade elementary aggregate. For 2022–2026 trends, older CAASPP file layouts must be supported first. The original percent-met notebook remains unchanged.
